# Dask
This is a tutorial to use the Dask cluster from Jupyter, without Prefect.

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *

init_demo()
init_dask_cluster_staging(scale=2)
init_dask_cluster_eopf(scale=2)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  

In [ ]:
# Other imports
import logging
import os
import sys
import time
from pathlib import Path

In [ ]:
# My local "./resources" folder contains a "dask_utils.py" module.
# I want to be able to use the same "import dask_utils" line on both client and workers.
# For this, I'm updating my PYTHONPATH.
sys.path.append("./resources")
import dask_utils

# Then I need to upload my local module to the dask workers
for client in dask_client_staging, dask_client_eopf:
    client.upload_file("./resources/dask_utils.py")

## 1. Implement the `Futures` tutorial: https://docs.dask.org/en/stable/futures.html
Note: this is how the `rs-server-staging` web service is using Dask.

In [ ]:
def inc(x, name):

    # From staging workers
    if name == "staging":
        # Just make sure that rs-server-staging is installed inside the dask workers.
        # NOTE: this import doesn't run the staging web service. 
        # It only imports its modules to be able to call the staging functions.
        # This is actually what the staging web service (that runs on another pod) is doing.
        from rs_server_staging.processors import processors

    # From eopf workers, we can also import the eopf modules
    else:
        from eopf.product.eo_product import EOProduct        
    
    # Note that this is run from a dask worker with a different IP than the client,
    # and that the workers also differ between the staging and eopf workers.
    logging.warning(
        f"Hello from {os.environ['HELLO_FROM']!r} {dask_utils.get_ip_address()!r} ({name})")    

    return x + 1

def add(x, y):
    return x + y

# Set environment variable for the dask workers
def set_dask_env():
    os.environ["HELLO_FROM"] = "dask"

# Print client (=jupyter or terminal) IP address
logging.warning(f"Hello from 'client' {dask_utils.get_ip_address()!r}")

# Test this for the staging and eopf client
for client, name in ((dask_client_staging, "staging"), (dask_client_eopf, "eopf")):
    print(f"\nTest {name!r}:")

    # Set environment variable for the dask workers
    client.run(set_dask_env)
    time.sleep(1)
    
    a = client.submit(inc, 10, name)  # calls inc(10) in background thread or process
    b = client.submit(inc, 20, name)  # calls inc(20) in background thread or process
    print(f"a: {a.result()}")
    print(f"b: {b.result()}")

    c = client.submit(add, a, b)  # calls add on the results of a and b
    print(f"c: {c.result()}")

    futures = client.map(inc, range(5), name=name)
    results = client.gather(futures)  # this can be faster
    print(results)

<div class="alert alert-info" role="alert">
Notes:

  1. Check in the logs that the client and dask clusters each run on **different** IP addresses.
      1. On kubernetes, you can run the `kubectl describe` command to check a pod IP address.
      1. In local mode, use: `docker inspect <container_id> | grep IPAddress`

## 2. `pip install` inside Dask workers

In [ ]:
# Test with any cluster
client = dask_client_eopf

In [ ]:
# Test if a module is installed inside the dask workers
def test_pip():
    import argh # yes this is a real module, see: https://pypi.org/project/argh/
    logging.warning(f" argh methods/attributes: {dir(argh)}")

# The first time you will test this in workers, it will fail
try:
    client.submit(test_pip).result()
except ModuleNotFoundError:
    print("'argh' is not yet installed in the workers ...")

# You can install it with: https://distributed.dask.org/en/stable/plugins.html#built-in-scheduler-plugins
from dask.distributed import PipInstall
plugin = PipInstall(packages=["argh"])
client.register_plugin(plugin)

# Now it will work.
client.submit(test_pip, pure=False).result() # IMPORTANT: use pure=False to disable cache
print("'argh' is now installed in the workers.")

In [ ]:
# Do the same with a wheel file. First download it.
whl_dir = "/tmp/emoji"
!rm -rf $whl_dir && mkdir -p $whl_dir && pip download --dest $whl_dir emoji
whl_file = os.listdir(whl_dir)[0]
whl_path = Path(whl_dir) / whl_file

# Then we'll upload and install it in the dask workers. 
# But it only works with .py, .egg or .zip
# See: https://distributed.dask.org/en/latest/api.html#distributed.Client.upload_file
# A .whl file is just a zip, so rename it.
whl_path = whl_path.rename(whl_path.with_suffix(".zip"))

In [ ]:
# Then do the same as before
def test_pip_whl():
    import emoji
    logging.warning(f" emoji methods/attributes: {dir(emoji)}")
try:
    client.submit(test_pip_whl, pure=False).result()
except ModuleNotFoundError:
    print("'emoji' is not yet installed in the workers ...")

# Upload the wheel/zip. It is automatically installed.
client.upload_file(str(whl_path))

client.submit(test_pip_whl, pure=False).result() # IMPORTANT: use pure=False to disable cache
print("'emoji' is now installed in the workers.")

## 3. Shutdown the dask clusters

In [ ]:
# You can scale the clusters to 0 workers
dask_gateway_staging.scale_cluster(dask_cluster_staging.name, 0)
dask_gateway_eopf.scale_cluster(dask_cluster_eopf.name, 0)

# Or shutdown the clusters
shutdown_dask_clusters(dask_gateway_staging, dask_cluster_staging.name)
shutdown_dask_clusters(dask_gateway_eopf, dask_cluster_eopf.name)

# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.